# Qwen3-32B · 개발 환자 356명 5-fold 비교

사용자가 지정한 임상 데이터와 분할의 train·validation 356명을 5-fold로 나누고 같은 fold의 사례는 제외합니다. 별도 test 89명은 사용하지 않습니다. train 표준화·사례 선택도 평가 fold를 제외하여 수행합니다. 이전 실험에서 이미 본 개발 데이터이므로 외부 검증이 아닙니다. 이 점수는 연구용입니다.

이 노트북은 진행 상태와 결과만 읽습니다. 개인별 정보는 비공개로 유지하세요.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

RESULT_DIR = Path("/root/.cache/std3_holter/llm_compare/run_20261102_cv5")
metadata_path = RESULT_DIR / "metadata.json"
if not metadata_path.exists():
    raise FileNotFoundError("실험이 아직 시작되지 않았습니다.")
meta = json.loads(metadata_path.read_text())
print("실험 상태:", meta["status"])
print("모델:", meta["model"])
print("조건:", ", ".join(meta["conditions"]))
if (RESULT_DIR / "predictions.csv").exists():
    predictions = pd.read_csv(RESULT_DIR / "predictions.csv")
    display(predictions.groupby(["condition", "status"]).size().rename("건수").reset_index())
if (RESULT_DIR / "metrics.json").exists():
    metrics = json.loads((RESULT_DIR / "metrics.json").read_text())
    display(pd.DataFrame({k: v["all_successful"] for k, v in metrics.items()}).T)
    print("세 조건 모두 성공한 동일 환자만 비교:")
    display(pd.DataFrame({k: v["paired_successful"] for k, v in metrics.items()}).T)

    print("조건별 fold 결과:")
    display(
        pd.DataFrame(
            [
                {"condition": k, "fold": fold, **values}
                for k, v in metrics.items()
                for fold, values in v["folds"].items()
            ]
        )
    )